In [ ]:
# 1. Clone the project and install requirements in this first cell.
import os, subprocess, sys, tempfile, getpass
from pathlib import Path
REPO_URL = "https://github.com/ParamShelar/diffusion-misspec-benchmark"  # @param {type:"string"}
if not REPO_URL.startswith("https://github.com/"):
    raise ValueError("Provide an HTTPS GitHub repository URL.")
REPO_DIR = Path("/content/ddpm")
if not REPO_DIR.exists():
    clone_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    attempt = subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], env=clone_env, capture_output=True, text=True)
    if attempt.returncode:
        # The repository is public, so this path is only reached if the clone fails
        # (network, rate limit, or a fork you have made private). The token is entered
        # invisibly, passed through a temporary askpass environment, never stored in the URL.
        token = getpass.getpass("Clone failed. GitHub token with read access (leave blank to abort): ")
        with tempfile.TemporaryDirectory() as temporary:
            askpass = Path(temporary) / "askpass.py"
            askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprint('x-access-token' if 'Username' in sys.argv[1] else os.environ['DDPM_GITHUB_TOKEN'])\n")
            askpass.chmod(0o700)
            clone_env.update(GIT_ASKPASS=str(askpass), DDPM_GITHUB_TOKEN=token)
            try:
                subprocess.run(["git", "-c", "credential.helper=", "clone", REPO_URL, str(REPO_DIR)], env=clone_env, check=True)
            finally:
                clone_env.pop("DDPM_GITHUB_TOKEN", None)
                del token
else:
    origin = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
    if origin.rstrip("/").removesuffix(".git") != REPO_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError("/content/ddpm belongs to a different repository.")
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


# OrganAMNIST DDPM training on Colab L4
Run the borrowed-CIFAR smoke pipeline first. This notebook resumes automatically from Drive. The default **90-minute budget includes training, monitoring, checkpointing and the quality gate within the training call**; downloads in setup cells are outside that compute budget. Forty-five minutes are reserved for the 2,000-sample quality gate. If the gate cannot finish in the budget, it reports FAIL and the research sweep stays blocked. The notebook does not promise that 90 minutes will yield a good prior.

The first cell is configured for `ParamShelar/diffusion-misspec-benchmark`, which is public. If the clone fails it falls back to asking for a GitHub token; the token is never stored in the clone URL or notebook source. Select **Runtime → Change runtime type → L4 GPU**.


In [ ]:
# 2. Mount persistent storage before creating any checkpoints, logs or samples.
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/ddpm-benchmark")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# 3. Verify the requested GPU before allocating the model.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime.")
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {name}; VRAM: {vram:.2f} GiB")
if "T4" in name:
    print("WARNING: T4 DOES NOT SUPPORT THE REQUIRED bf16 TRAINING. Select L4.")
assert "L4" in name, f"Expected L4, got {name}"
assert torch.cuda.is_bf16_supported(), "Native bf16 required"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True


In [ ]:
# 4. Configuration: all checkpoint, log, sample and result paths are on Drive.
import yaml
import argparse, shlex
TRAIN_ARGS = "--max-minutes 90"  # @param {type:"string"}
# For a two-update notebook smoke run: --max-minutes 2 --smoke
# Compilation is deliberately opt-in: append --compile
parser = argparse.ArgumentParser()
parser.add_argument("--max-minutes", type=float, default=90)
parser.add_argument("--smoke", action="store_true")
parser.add_argument("--compile", action="store_true")
args = parser.parse_args(shlex.split(TRAIN_ARGS))
config = yaml.safe_load(Path("configs/colab.yaml").read_text())
experiment = "organamnist64-smoke" if args.smoke else "organamnist64"
config["output"] = str(DRIVE_ROOT / experiment)
config["quality"] = {"batch_size": 16}
config["training"] = {
    "checkpoint_dir": str(DRIVE_ROOT / experiment / "checkpoints"),
    "batch_size": 128,
    "quality_reserve_minutes": 45,
}
config["smoke"] = args.smoke
Path(config["output"]).mkdir(parents=True, exist_ok=True)
config_path = Path(config["output"]) / "training_config.yaml"
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print("Persistent experiment:", config["output"])


In [ ]:
# 5. Automatically cache the official data and Inception weights before timing.
# The data/cache may live on Colab disk; all irreplaceable training outputs use Drive.
from src.data.datasets import from_config
from src.metrics.quality import InceptionFeatures
os.environ["MPLCONFIGDIR"] = "/content/.cache/matplotlib"
torch.hub.set_dir("/content/.cache/torch")
train = from_config(config, "train")
validation = from_config(config, "val")
print("Official train/validation counts:", len(train), len(validation))
print("GPU uint8 training tensor will occupy", train.numel()/2**20, "MiB")
del train, validation
features = InceptionFeatures("cpu")
del features


In [ ]:
# 6. Auto-resume; only EMA weights are sampled/evaluated.
# Latest usable Drive checkpoint restores model, EMA, optimizer, step and RNG states.
from src.models.train import train_colab
result = train_colab(config, max_minutes=args.max_minutes, smoke=args.smoke, compile_model=args.compile)
print("Explicit quality-gate status:", result["status"])


In [ ]:
# 7. Inspect persistent artifacts and the training summary.
import json
from IPython.display import display, Image
summary = json.loads((Path(config["output"]) / "training_summary.json").read_text())
print(json.dumps({k:v for k,v in summary.items() if k != "gate"}, indent=2))
for name in ["prior_samples.png", "noise_then_denoise.png"]:
    path = Path(config["output"]) / name
    if path.exists():
        display(Image(filename=str(path)))
print("Rerun this notebook after a disconnect to continue from the latest checkpoint.")


## Run the full benchmark after a scientific PASS
The following cell performs the research sweep and produces every figure using the EMA checkpoint that passed the gate. It is a separate experiment after the training budget and may take hours. Smoke results cannot enable it. All outputs remain on Drive.


In [ ]:
# 8. Research sweep (the script independently revalidates the quality gate).
if summary["gate"]["status"] != "PASS":
    raise RuntimeError("Research sweep blocked: this prior has not passed the full quality gate.")
sweep_config = summary["sweep_config"]
subprocess.run([sys.executable, "scripts/run_sweep.py", "--config", sweep_config], check=True)
subprocess.run([sys.executable, "scripts/make_figures.py", "--config", sweep_config], check=True)
